# Dice Dataset Generator

Generates a synthetic dataset of dice images with structured symbol sequences and natural language labels.

**Three-stage pipeline:**
```
DiceSymbol  →  sentence   (describe_scene)
DiceSymbol  →  image      (render_scene_image)
```
Both sentence and image are derived from the `DiceSymbol`

**Notebook order:**
1. Imports & Configuration
2. Data Structures (`DiceSymbol`, `Die`, `DiceScene`)
3. Random Scene Generator
4. Text Label Generator
5. Image Renderer
6. Dataset Generator
7. PyTorch Dataset & DataLoader

## 1. Imports & Configuration

In [1]:
import random
import os
import csv
import shutil
import logging
from dataclasses import dataclass, field

import numpy as np
from PIL import Image, ImageDraw

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Global config
CANVAS_SIZE = 320
MARGIN      = 20
MIN_DIE_SIZE = 60
MAX_DIE_SIZE = 100

# Discrete size categories and their pixel ranges
SIZE_CATEGORIES = {
    "small":  (60, 69),
    "medium": (70, 89),
    "large":  (90, 100),
}
VALID_SIZES = list(SIZE_CATEGORIES.keys())

COLOUR_MAP = {
    "white":  (255, 255, 255),
    "red":    (255, 220, 220),
    "blue":   (220, 235, 255),
    "green":  (220, 255, 220),
    "yellow": (255, 245, 200),
    "purple": (235, 220, 255),
    "peach":  (255, 230, 210),
}
VALID_COLOURS = list(COLOUR_MAP.keys())

DIE_OUTLINE = (40, 40, 40)
PIP_COLOR   = (20, 20, 20)

print("Imports OK")
print(f"Size categories: {SIZE_CATEGORIES}")
print(f"Colours: {VALID_COLOURS}")

Imports OK
Size categories: {'small': (60, 69), 'medium': (70, 89), 'large': (90, 100)}
Colours: ['white', 'red', 'blue', 'green', 'yellow', 'purple', 'peach']


## 2. Data Structures

`DiceSymbol` is the canonical intermediate representation, a structured, discrete description of the scene. Both the sentence and the image are derived from it, never from each other directly.

In [2]:
def discretise_size(px: int) -> str:
    """Map a pixel size to a discrete size category."""
    for label, (lo, hi) in SIZE_CATEGORIES.items():
        if lo <= px <= hi:
            return label
    raise ValueError(f"Pixel size {px} outside expected range {MIN_DIE_SIZE}-{MAX_DIE_SIZE}.")


def size_to_pixels(size_label: str) -> int:
    """Sample a random pixel size from within a size category."""
    if size_label not in SIZE_CATEGORIES:
        raise ValueError(f"Unknown size label '{size_label}'. Must be one of {VALID_SIZES}.")
    lo, hi = SIZE_CATEGORIES[size_label]
    return random.randint(lo, hi)


@dataclass
class DiceSymbol:
    """
    Canonical structured representation of a dice scene.
    This is the ground truth the neural network learns to predict.
    All fields are discrete symbols.
    """
    num_dice: int          # 1, 2, or 3
    values:   list         # e.g. [3, 5]
    colours:  list         # e.g. ["blue", "red"]
    sizes:    list         # e.g. ["large", "small"]

    def __post_init__(self):
        if not (1 <= self.num_dice <= 3):
            raise ValueError(f"num_dice must be 1-3, got {self.num_dice}.")
        if len(self.values) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} values, got {len(self.values)}.")
        if len(self.colours) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} colours, got {len(self.colours)}.")
        if len(self.sizes) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} sizes, got {len(self.sizes)}.")
        for v in self.values:
            if not (1 <= v <= 6):
                raise ValueError(f"Die value must be 1-6, got {v}.")
        for c in self.colours:
            if c not in VALID_COLOURS:
                raise ValueError(f"Invalid colour '{c}'. Must be one of {VALID_COLOURS}.")
        for s in self.sizes:
            if s not in VALID_SIZES:
                raise ValueError(f"Invalid size '{s}'. Must be one of {VALID_SIZES}.")

    def __str__(self):
        dice_strs = [
            f"Die(value={v}, colour={c}, size={s})"
            for v, c, s in zip(self.values, self.colours, self.sizes)
        ]
        return f"DiceSymbol({self.num_dice} dice: {' | '.join(dice_strs)})"


@dataclass
class Die:
    """Rendering-level die — extends DiceSymbol with pixel-level position/angle."""
    value:  int
    x:      int
    y:      int
    size:   int   = 80
    angle:  float = 0.0
    colour: str   = "white"

    def __post_init__(self):
        if self.colour not in COLOUR_MAP:
            raise ValueError(f"Invalid colour '{self.colour}'. Must be one of {VALID_COLOURS}.")
        if not (1 <= self.value <= 6):
            raise ValueError(f"Die value must be 1-6, got {self.value}.")

    @property
    def size_label(self) -> str:
        """Return the discrete size category for this die."""
        return discretise_size(self.size)


@dataclass
class DiceScene:
    dice: list = field(default_factory=list)

    def __post_init__(self):
        self._validate()

    def _validate(self):
        if not (1 <= len(self.dice) <= 3):
            raise ValueError(f"A scene must contain 1-3 dice, got {len(self.dice)}.")
        for die in self.dice:
            if not (MIN_DIE_SIZE <= die.size <= MAX_DIE_SIZE):
                raise ValueError(f"Die size must be {MIN_DIE_SIZE}-{MAX_DIE_SIZE}, got {die.size}.")
        positions = [(die.x, die.y) for die in self.dice]
        if len(positions) != len(set(positions)):
            raise ValueError("Two dice cannot have exactly the same top-left position.")

    def to_symbol(self) -> DiceSymbol:
        """Extract the canonical DiceSymbol from this scene."""
        return DiceSymbol(
            num_dice=len(self.dice),
            values=[d.value for d in self.dice],
            colours=[d.colour for d in self.dice],
            sizes=[d.size_label for d in self.dice],
        )

    def count(self) -> int:
        return len(self.dice)

    def values(self) -> list:
        return [die.value for die in self.dice]

    def __str__(self) -> str:
        return " | ".join(
            [f"Die(value={d.value}, colour={d.colour}, size={d.size_label}, x={d.x}, y={d.y})" for d in self.dice]
        )

## 3. Random Scene Generator

In [3]:
def _overlap(d1: Die, d2: Die, padding: int = 10) -> bool:
    """Return True if two dice bounding boxes overlap (with padding)."""
    return not (
        d1.x + d1.size + padding <= d2.x or
        d2.x + d2.size + padding <= d1.x or
        d1.y + d1.size + padding <= d2.y or
        d2.y + d2.size + padding <= d1.y
    )


def _is_valid_placement(new_die: Die, placed_dice: list, padding: int = 10) -> bool:
    """Return True if new_die does not overlap any already-placed dice."""
    return all(not _overlap(new_die, die, padding=padding) for die in placed_dice)


def random_symbol(num_dice: int = None) -> DiceSymbol:
    """
    Generate a random DiceSymbol — the structured ground truth.
    This is the first stage of the pipeline.
    """
    if num_dice is None:
        num_dice = random.randint(1, 3)
    if not (1 <= num_dice <= 3):
        raise ValueError(f"num_dice must be 1-3, got {num_dice}.")

    return DiceSymbol(
        num_dice=num_dice,
        values=[random.randint(1, 6) for _ in range(num_dice)],
        colours=[random.choice(VALID_COLOURS) for _ in range(num_dice)],
        sizes=[random.choice(VALID_SIZES) for _ in range(num_dice)],
    )


def symbol_to_scene(
    symbol: DiceSymbol,
    canvas_size: int = CANVAS_SIZE,
    max_attempts: int = 200
) -> DiceScene:
    """
    Render a DiceSymbol into a DiceScene by assigning pixel-level
    positions and angles. This is the second stage of the pipeline.
    """
    dice = []

    for i in range(symbol.num_dice):
        placed = False

        for _ in range(max_attempts):
            size = size_to_pixels(symbol.sizes[i])
            max_x = canvas_size - size - MARGIN
            max_y = canvas_size - size - MARGIN

            if max_x < MARGIN or max_y < MARGIN:
                raise ValueError(
                    f"Canvas too small ({canvas_size}px) for die of size {size}px "
                    f"with margin {MARGIN}px."
                )

            x     = random.randint(MARGIN, max_x)
            y     = random.randint(MARGIN, max_y)
            angle = random.uniform(-15, 15)

            candidate = Die(
                value=symbol.values[i],
                x=x, y=y,
                size=size,
                angle=angle,
                colour=symbol.colours[i]
            )

            if _is_valid_placement(candidate, dice):
                dice.append(candidate)
                placed = True
                break

        if not placed:
            raise RuntimeError(
                f"Could not place die {i+1}/{symbol.num_dice} without overlap after "
                f"{max_attempts} attempts. Try a larger canvas or fewer dice."
            )

    return DiceScene(dice=dice)


def random_scene(num_dice: int = None) -> tuple:
    """
    Convenience wrapper: generate a random symbol and convert it to a scene.
    Returns (DiceSymbol, DiceScene) so both are always available.
    """
    symbol = random_symbol(num_dice)
    scene  = symbol_to_scene(symbol)
    return symbol, scene


## 4. Text Label Generator

Sentences are generated from `DiceSymbol` fields, including size. Multiple templates ensure variety.

In [4]:
NUMBER_WORDS = {
    1: "one", 2: "two", 3: "three",
    4: "four", 5: "five", 6: "six",
}
COUNT_WORDS = {
    1: "one die", 2: "two dice", 3: "three dice",
}


def _value_word(v: int) -> str:
    if v not in NUMBER_WORDS:
        raise ValueError(f"Die value must be 1-6, got {v}.")
    return NUMBER_WORDS[v]


def _join_phrases(items: list) -> str:
    """Join a list of strings naturally: 'a, b and c'."""
    if not items:
        raise ValueError("Cannot join an empty list.")
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return items[0] + " and " + items[1]
    return ", ".join(items[:-1]) + " and " + items[-1]


def describe_symbol(symbol: DiceSymbol) -> str:
    """
    Generate a natural language sentence from a DiceSymbol.
    Includes value, colour, AND size so all symbol fields appear in the text.
    """
    value_words = [_value_word(v) for v in symbol.values]
    count       = symbol.num_dice
    colours     = symbol.colours
    sizes       = symbol.sizes

    all_same_colour = len(set(colours)) == 1
    all_same_size   = len(set(sizes)) == 1

    template = random.randint(1, 4)

    if count == 1:
        s, c, v = sizes[0], colours[0], value_words[0]
        templates = [
            f"There is one {s} {c} die showing {v}.",
            f"The image shows one {s} {c} die with value {v}.",
            f"This picture contains a {s} {c} die displaying {v}.",
            f"A {s} {c} die is shown with a value of {v}.",
        ]

    elif all_same_colour and all_same_size:
        s, c = sizes[0], colours[0]
        vals = _join_phrases(value_words)
        templates = [
            f"There are {count} {s} {c} dice showing {vals}.",
            f"The image shows {count} {s} {c} dice with values {vals}.",
            f"This picture contains {count} {s} {c} dice displaying {vals}.",
            f"A scene with {count} {s} {c} dice is shown, displaying {vals}.",
        ]

    else:
        # Describe each die individually with its own size, colour, and value
        parts = [
            f"a {s} {c} die showing {v}"
            for s, c, v in zip(sizes, colours, value_words)
        ]
        joined = _join_phrases(parts)
        templates = [
            f"The image shows {joined}.",
            f"This picture contains {joined}.",
            f"There are {joined} in the scene.",
            f"The scene contains {joined}.",
        ]

    return templates[template - 1]


## 5. Image Renderer

Renders a `DiceScene` into a PIL image. Note: the renderer takes a `DiceScene` (which has pixel-level detail), not a `DiceSymbol` directly.

In [5]:
PIP_MAP = {
    1: ["mc"],
    2: ["tl", "br"],
    3: ["tl", "mc", "br"],
    4: ["tl", "tr", "bl", "br"],
    5: ["tl", "tr", "mc", "bl", "br"],
    6: ["tl", "tr", "ml", "mr", "bl", "br"],
}


def _random_background() -> tuple:
    """Return a slightly varied light background colour."""
    return tuple(random.randint(235, 250) for _ in range(3))


def _pip_positions(x: int, y: int, size: int) -> dict:
    """Return standard pip anchor positions for a die face."""
    left  = x + size * 0.25
    cx    = x + size * 0.5
    right = x + size * 0.75
    top   = y + size * 0.25
    cy    = y + size * 0.5
    bot   = y + size * 0.75
    return {
        "tl": (left, top),  "tc": (cx, top),   "tr": (right, top),
        "ml": (left, cy),   "mc": (cx, cy),     "mr": (right, cy),
        "bl": (left, bot),  "bc": (cx, bot),    "br": (right, bot),
    }


def _draw_pip(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: int = 6):
    draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=PIP_COLOR)


def _draw_die(img: Image.Image, die: Die):
    """Draw a single die onto the canvas image."""
    if die.value not in PIP_MAP:
        raise ValueError(f"Cannot draw die with value {die.value}. Must be 1-6.")
    if die.colour not in COLOUR_MAP:
        raise ValueError(f"Unknown colour '{die.colour}'.")

    size = die.size
    pad  = 20
    tile = size + pad * 2

    die_img = Image.new("RGBA", (tile, tile), (0, 0, 0, 0))
    draw    = ImageDraw.Draw(die_img)

    x0, y0 = pad, pad
    x1, y1 = pad + size, pad + size

    # Soft shadow
    so = 4
    draw.rounded_rectangle(
        (x0 + so, y0 + so, x1 + so, y1 + so),
        radius=12, fill=(0, 0, 0, 50)
    )
    # Die body
    draw.rounded_rectangle(
        (x0, y0, x1, y1),
        radius=12,
        fill=COLOUR_MAP[die.colour],
        outline=DIE_OUTLINE,
        width=3
    )
    # Pips
    pos   = _pip_positions(x0, y0, size)
    pip_r = max(4, size // 12)
    for key in PIP_MAP[die.value]:
        _draw_pip(draw, *pos[key], r=pip_r)

    rotated = die_img.rotate(die.angle, expand=True, resample=Image.Resampling.BICUBIC)
    paste_x = die.x - (rotated.width  - size) // 2
    paste_y = die.y - (rotated.height - size) // 2
    img.paste(rotated, (paste_x, paste_y), rotated)


def render_scene_image(
    scene: DiceScene,
    canvas_size: int = CANVAS_SIZE,
    background_color: tuple = None
) -> Image.Image:
    """Render a DiceScene into a PIL RGB image."""
    if background_color is None:
        background_color = _random_background()
    if len(background_color) != 3 or not all(0 <= c <= 255 for c in background_color):
        raise ValueError(f"background_color must be an RGB tuple with values 0-255.")

    img = Image.new("RGB", (canvas_size, canvas_size), background_color)
    for die in scene.dice:
        _draw_die(img, die)
    return img

## 6. Dataset Generator

Generates `total_samples` examples. Each sample stores:
- The rendered image (PNG)
- The natural language sentence
- The explicit symbol fields (`sym_num_dice`, `sym_values`, `sym_colours`, `sym_sizes`)

The symbol fields are the ground truth for the neural network.

In [6]:
def make_clean_dir(path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def split_counts(total: int, train_ratio: float = 0.7, val_ratio: float = 0.15, test_ratio: float = 0.15):
    if not abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6:
        raise ValueError("Split ratios must sum to 1.0.")
    train_n = int(total * train_ratio)
    val_n   = int(total * val_ratio)
    test_n  = total - train_n - val_n
    return train_n, val_n, test_n


def generate_dataset(
    output_dir: str = "dataset",
    total_samples: int = 6000,
    seed: int = 42
):
    """
    Generate the full dataset.

    Pipeline per sample:
        1. random_symbol()       → DiceSymbol  (ground truth)
        2. describe_symbol()     → sentence    (derived from symbol)
        3. symbol_to_scene()     → DiceScene   (derived from symbol)
        4. render_scene_image()  → PIL image   (derived from scene)
    """
    if total_samples < 3:
        raise ValueError("total_samples must be at least 3.")

    random.seed(seed)
    np.random.seed(seed)

    make_clean_dir(output_dir)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)

    train_n, val_n, test_n = split_counts(total_samples)
    split_plan = ["train"] * train_n + ["val"] * val_n + ["test"] * test_n
    random.shuffle(split_plan)

    split_rows = {"train": [], "val": [], "test": []}
    errors = []

    for i in range(total_samples):
        try:
            # Stage 1: symbol (ground truth)
            symbol = random_symbol()
            # Stage 2: sentence (from symbol)
            sentence = describe_symbol(symbol)
            # Stage 3: scene (from symbol)
            scene = symbol_to_scene(symbol)
            # Stage 4: image (from scene)
            image = render_scene_image(scene)
        except Exception as e:
            logger.warning(f"Sample {i} failed to generate: {e}")
            errors.append(i)
            continue

        split          = split_plan[i]
        image_filename = f"sample_{i:05d}.png"
        image_path     = os.path.join(output_dir, split, "images", image_filename)

        try:
            image.save(image_path)
        except Exception as e:
            logger.warning(f"Sample {i}: failed to save image: {e}")
            errors.append(i)
            continue

        split_rows[split].append({
            "image":        image_filename,
            "text":         sentence,
            # Explicit symbol fields — ground truth for the neural network
            "sym_num_dice": symbol.num_dice,
            "sym_values":   " ".join(str(v) for v in symbol.values),
            "sym_colours":  " ".join(symbol.colours),
            "sym_sizes":    " ".join(symbol.sizes),
        })

        if (i + 1) % 1000 == 0:
            logger.info(f"Generated {i + 1}/{total_samples} samples...")

    fieldnames = ["image", "text", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes"]
    for split in ["train", "val", "test"]:
        csv_path = os.path.join(output_dir, split, "labels.csv")
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(split_rows[split])

    meta_path = os.path.join(output_dir, "dataset_info.txt")
    with open(meta_path, "w", encoding="utf-8") as f:
        f.write(f"total_samples: {total_samples}\n")
        f.write(f"successfully_generated: {total_samples - len(errors)}\n")
        f.write(f"failed: {len(errors)}\n")
        f.write(f"seed: {seed}\n")
        f.write(f"train_count: {len(split_rows['train'])}\n")
        f.write(f"val_count: {len(split_rows['val'])}\n")
        f.write(f"test_count: {len(split_rows['test'])}\n")
        f.write("split_ratio: 70/15/15\n")
        f.write(f"symbol_fields: sym_num_dice, sym_values, sym_colours, sym_sizes\n")
        f.write(f"size_categories: {SIZE_CATEGORIES}\n")

    print("\nDataset generated successfully.")
    print(f"  Train: {len(split_rows['train'])}, Val: {len(split_rows['val'])}, Test: {len(split_rows['test'])}")
    if errors:
        print(f"  Warnings: {len(errors)} samples failed (see logs).")
    print(f"  Saved to: {output_dir}/")
    print(f"  Symbol fields: sym_num_dice, sym_values, sym_colours, sym_sizes")


generate_dataset(output_dir="dataset", total_samples=6000, seed=42)

INFO: Generated 1000/6000 samples...
INFO: Generated 2000/6000 samples...
INFO: Generated 3000/6000 samples...
INFO: Generated 4000/6000 samples...
INFO: Generated 5000/6000 samples...
INFO: Generated 6000/6000 samples...



Dataset generated successfully.
  Train: 4181, Val: 899, Test: 898
  Warnings: 22 samples failed (see logs).
  Saved to: dataset/
  Symbol fields: sym_num_dice, sym_values, sym_colours, sym_sizes


## 7. PyTorch Dataset & DataLoader

The dataset returns both the text and the explicit symbol tensors. The symbol tensors (`sym_num_dice`, `sym_values`, `sym_colours`, `sym_sizes`) are the direct prediction targets for the neural network.

In [7]:
# Index maps for encoding symbol fields as class indices
COLOUR_TO_IDX = {c: i for i, c in enumerate(VALID_COLOURS)}
SIZE_TO_IDX   = {s: i for i, s in enumerate(VALID_SIZES)}

print("Colour index map:", COLOUR_TO_IDX)
print("Size index map:  ", SIZE_TO_IDX)


class DiceDataset(Dataset):
    """
    PyTorch Dataset for the synthetic dice dataset.

    Each item returns:
        image       — float32 tensor [3, H, W]
        label dict  — containing text and symbol tensors

    Symbol tensors (ground truth for the neural network):
        sym_num_dice  — scalar long tensor (1-3)
        sym_values    — long tensor [3], padded with 0
        sym_colours   — long tensor [3], padded with -1
        sym_sizes     — long tensor [3], padded with -1
    """

    VALID_SPLITS = ("train", "val", "test")

    def __init__(self, root_dir: str, split: str = "train", transform=None):
        if split not in self.VALID_SPLITS:
            raise ValueError(f"split must be one of {self.VALID_SPLITS}, got '{split}'.")

        self.root_dir  = root_dir
        self.split     = split
        self.image_dir = os.path.join(root_dir, split, "images")
        self.csv_path  = os.path.join(root_dir, split, "labels.csv")

        if not os.path.isdir(self.image_dir):
            raise FileNotFoundError(
                f"Image directory not found: '{self.image_dir}'.\n"
                "Have you run generate_dataset() first?"
            )
        if not os.path.isfile(self.csv_path):
            raise FileNotFoundError(
                f"Labels file not found: '{self.csv_path}'.\n"
                "Have you run generate_dataset() first?"
            )

        self.samples = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append(row)

        if len(self.samples) == 0:
            raise RuntimeError(f"No samples found in {self.csv_path}.")

        self.transform = transform or v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])

        logger.info(f"DiceDataset ({split}): {len(self.samples)} samples loaded.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        if not (0 <= idx < len(self.samples)):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.samples)}.")

        row      = self.samples[idx]
        img_path = os.path.join(self.image_dir, row["image"])

        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image file missing: '{img_path}'.")

        try:
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)
        except Exception as e:
            raise RuntimeError(f"Failed to load/transform image at '{img_path}': {e}") from e

        # Parse symbol fields
        try:
            num_dice = int(row["sym_num_dice"])
            values   = list(map(int,   row["sym_values"].split()))
            colours  = row["sym_colours"].split()
            sizes    = row["sym_sizes"].split()
        except (ValueError, KeyError) as e:
            raise ValueError(f"Malformed symbol fields in row {idx}: {e}") from e

        # Validate
        if not (1 <= num_dice <= 3):
            raise ValueError(f"sym_num_dice out of range in row {idx}: {num_dice}")
        if not all(1 <= v <= 6 for v in values):
            raise ValueError(f"sym_values out of range in row {idx}: {values}")
        if not all(c in COLOUR_TO_IDX for c in colours):
            raise ValueError(f"Unknown colour in row {idx}: {colours}")
        if not all(s in SIZE_TO_IDX for s in sizes):
            raise ValueError(f"Unknown size in row {idx}: {sizes}")

        # Pad to length 3 (max dice)
        padded_values  = values  + [0]  * (3 - len(values))
        padded_colours = [COLOUR_TO_IDX[c] for c in colours] + [-1] * (3 - len(colours))
        padded_sizes   = [SIZE_TO_IDX[s]   for s in sizes]   + [-1] * (3 - len(sizes))

        label = {
            "text":         row["text"],
            "sym_num_dice": torch.tensor(num_dice,       dtype=torch.long),
            "sym_values":   torch.tensor(padded_values,  dtype=torch.long),
            "sym_colours":  torch.tensor(padded_colours, dtype=torch.long),
            "sym_sizes":    torch.tensor(padded_sizes,   dtype=torch.long),
        }

        return image, label

Colour index map: {'white': 0, 'red': 1, 'blue': 2, 'green': 3, 'yellow': 4, 'purple': 5, 'peach': 6}
Size index map:   {'small': 0, 'medium': 1, 'large': 2}


In [8]:
def dice_collate_fn(batch):
    """Custom collate to handle variable-length string 'text' field."""
    images = torch.stack([item[0] for item in batch])
    labels = {
        "text":         [item[1]["text"]         for item in batch],
        "sym_num_dice": torch.stack([item[1]["sym_num_dice"] for item in batch]),
        "sym_values":   torch.stack([item[1]["sym_values"]   for item in batch]),
        "sym_colours":  torch.stack([item[1]["sym_colours"]  for item in batch]),
        "sym_sizes":    torch.stack([item[1]["sym_sizes"]     for item in batch]),
    }
    return images, labels


try:
    train_dataset = DiceDataset("dataset", split="train")
    val_dataset   = DiceDataset("dataset", split="val")
    test_dataset  = DiceDataset("dataset", split="test")

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0, collate_fn=dice_collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0, collate_fn=dice_collate_fn)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0, collate_fn=dice_collate_fn)

    print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

    for imgs, labels in train_loader:
        print("Image shape:        ", imgs.shape)                    # [32, 3, 320, 320]
        print("sym_num_dice shape: ", labels["sym_num_dice"].shape)  # [32]
        print("sym_values shape:   ", labels["sym_values"].shape)    # [32, 3]
        print("sym_colours shape:  ", labels["sym_colours"].shape)   # [32, 3]
        print("sym_sizes shape:    ", labels["sym_sizes"].shape)     # [32, 3]
        print("Example text:       ", labels["text"][0])
        print("Example num_dice:   ", labels["sym_num_dice"][0].item())
        print("Example values:     ", labels["sym_values"][0].tolist())
        print("Example colours:    ", labels["sym_colours"][0].tolist())
        print("Example sizes:      ", labels["sym_sizes"][0].tolist())
        break

except FileNotFoundError as e:
    print(f"Dataset not found. Run generate_dataset() first.\nDetails: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")
    raise

INFO: DiceDataset (train): 4181 samples loaded.
INFO: DiceDataset (val): 899 samples loaded.
INFO: DiceDataset (test): 898 samples loaded.


Train batches: 131 | Val: 29 | Test: 29
Image shape:         torch.Size([32, 3, 320, 320])
sym_num_dice shape:  torch.Size([32])
sym_values shape:    torch.Size([32, 3])
sym_colours shape:   torch.Size([32, 3])
sym_sizes shape:     torch.Size([32, 3])
Example text:        A medium red die is shown with a value of three.
Example num_dice:    1
Example values:      [3, 0, 0]
Example colours:     [1, -1, -1]
Example sizes:       [1, -1, -1]
